# @tool과 Ch4 함수 재사용

검증한 조회 함수에 도구 인터페이스를 더합니다.

## 준비

공개 CSV와 Ch4 함수 원문을 사용합니다.

In [1]:
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = Path("data/orders.csv")
assert data_path.exists(), "노트북은 labs/part5/ch19-model-tools에서 실행합니다."
print("Python 실습 준비 완료")

Python 실습 준비 완료


In [2]:
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")

## 도구 정의

In [3]:
from langchain.tools import tool
@tool
def list_orders_by_status(status: str) -> list[dict[str, Any]]:
    """주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다."""
    return fetch_orders(data_path, status=status)

## 같은 값 비교

In [4]:
direct = fetch_orders(data_path, status="cancelled")
wrapped = list_orders_by_status.invoke({"status": "cancelled"})
print({"same": direct == wrapped, "rows": wrapped})

{'same': True, 'rows': [{'order_id': 'O104', 'customer_id': 'C003', 'status': 'cancelled', 'amount': 9000}]}


## 실패 재현

In [5]:
try:
    list_orders_by_status("paid")
except TypeError as error:
    print(type(error).__name__, str(error))

TypeError 'StructuredTool' object is not callable


## 수정

In [6]:
result = list_orders_by_status.invoke({"status": "paid"})
print({"count": len(result), "first_id": result[0]["order_id"]})

{'count': 2, 'first_id': 'O101'}
